In [ ]:
import os
def savePic( filename ):
    # Ensure the output directory exists
    output_dir = os.path.join(os.getcwd(), 'Picture')
    os.makedirs(output_dir, exist_ok=True)
    # Save to the "Picture" folder
    filepath = os.path.join(output_dir, filename)
    # print(filepath)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
# from google.colab import drive

# Import TDA utilities
import gudhi
from gudhi.representations import Landscape
import gudhi.representations
from ripser import Rips
import plotly.graph_objects as go
from persim.landscapes import PersLandscapeExact
from persim.landscapes import plot_landscape_simple
from persim.landscapes import PersistenceLandscaper
from persim import plot_diagrams
from tadasets import torus, sphere
from matplotlib.patches import Polygon, FancyArrowPatch
# Import Scikit-Learn tools
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn import metrics
from sklearn.model_selection import train_test_split
from mpl_toolkits.mplot3d import Axes3D # Required for 3D plots
from matplotlib.patches import Circle, Ellipse, FancyArrowPatch


In [ ]:
print("Generating 100 points on sphere and torus surfaces, then adding noise...")

# --- Parameters ---
num_points = 200 # Number of points to generate for each surface
R_sphere = 1.0  # Radius of the sphere

R_major = 2.0   # Major radius of the torus
r_minor = 0.8   # Minor radius of the torus

# Noise scale: This is crucial for making the shapes indistinguishable.
# A scale of 1.0 means the standard deviation of the noise is 1 unit.
# Since the sphere has a radius of 1 and the torus's features are around 0.8-2.8 units,
# this level of noise will significantly blur the original shapes.
noise_scale = .50 # Standard deviation of the Gaussian noise

# --- Sphere Point Generation ---
# Generate random spherical coordinates for the sphere
# u: azimuthal angle (0 to 2*pi)
# v: polar angle (0 to pi)
u_sphere_rand = np.random.uniform(0, 2 * np.pi, num_points)
v_sphere_rand = np.random.uniform(0, np.pi, num_points)

# Convert spherical coordinates to Cartesian coordinates
x_sphere_orig = R_sphere * np.sin(v_sphere_rand) * np.cos(u_sphere_rand)
y_sphere_orig = R_sphere * np.sin(v_sphere_rand) * np.sin(u_sphere_rand)
z_sphere_orig = R_sphere * np.cos(v_sphere_rand)

# Stack original coordinates for easier noise addition
sphere_original_points = np.stack([x_sphere_orig, y_sphere_orig, z_sphere_orig], axis=-1)

# Add Gaussian noise
noise_sphere = np.random.normal(loc=0.0, scale=noise_scale, size=sphere_original_points.shape)
cloud_sphere = sphere_original_points + noise_sphere
print(f"Generated {num_points} noisy points for the sphere.")

# --- Torus Point Generation ---
# Generate random parametric coordinates for the torus
# theta: angle around the tube (0 to 2*pi)
# phi: angle around the torus's central axis (0 to 2*pi)
theta_torus_rand = np.random.uniform(0, 2 * np.pi, num_points)
phi_torus_rand = np.random.uniform(0, 2 * np.pi, num_points)

# Parametric equations for a torus
x_torus_orig = (R_major + r_minor * np.cos(theta_torus_rand)) * np.cos(phi_torus_rand)
y_torus_orig = (R_major + r_minor * np.cos(theta_torus_rand)) * np.sin(phi_torus_rand)
z_torus_orig = r_minor * np.sin(theta_torus_rand)

# Stack original coordinates
torus_original_points = np.stack([x_torus_orig, y_torus_orig, z_torus_orig], axis=-1)

# Add Gaussian noise
noise_torus = np.random.normal(loc=0.0, scale=noise_scale, size=torus_original_points.shape)
cloud_torus = torus_original_points + noise_torus


# --- Plotting ---
fig = plt.figure(figsize=(14, 7), dpi=300) 
fig.set_size_inches(7.3, 4.1)
# Create a figure with two subplots

# Subplot 1: Noisy Sphere Points
ax2 = fig.add_subplot(1,2,1, projection='3d') # 1 row, 2 columns, first plot
ax2.scatter(cloud_sphere[:, 0], cloud_sphere[:, 1], cloud_sphere[:, 2], color='blue', alpha=0.7, s=10, label=f'Point Cloud {num_points} ')
# ax1.set_title(f'Noisy Point Cloud (Sphere Origin)')
ax2.set_xlabel('X', labelpad=-10)
ax2.set_ylabel('Y', labelpad=-10)
ax2.set_zlabel('Z', labelpad=-10)
ax2.set_xticklabels([])
ax2.set_yticklabels([])
ax2.set_zticklabels([])
# Set equal aspect ratio to prevent distortion
ax2.set_box_aspect([1, 1, 1])
ax2.legend(fontsize=12, prop={'weight': 'bold'})
ax2.grid(True)

# Subplot 2: Noisy Torus Points
ax1 = fig.add_subplot(122, projection='3d') # 1 row, 2 columns, second plot
ax1.scatter(cloud_torus[:, 0], cloud_torus[:, 1], cloud_torus[:, 2], color='red', alpha=0.7, s=10, label=f'Point Cloud {num_points}')
# ax2.set_title(f'Noisy Point Cloud (Torus Origin)')
ax1.set_xlabel('X', labelpad=-10)
ax1.set_ylabel('Y', labelpad=-10)
ax1.set_zlabel('Z', labelpad=-10)
ax1.set_xticklabels([])
ax1.set_yticklabels([])
ax1.set_zticklabels([])
# Set equal aspect ratio
ax1.set_box_aspect([1, 1, 1])
ax1.legend( fontsize=12, prop={'weight': 'bold'})
ax1.grid(True)
savePic("100 point clouds")
plt.tight_layout() # Adjust subplot parameters for a tight layout

plt.show()

# Disable LaTeX to avoid the dvipng warning
plt.rcParams['text.usetex'] = False


def get_tda_data(cloud):
    rips = gudhi.RipsComplex(points=cloud)
    st = rips.create_simplex_tree(max_dimension=2)
    diag = st.persistence()
    h1_intervals = st.persistence_intervals_in_dimension(1)
    # Filter infinite values for landscape/axis calculations
    h1_clean = h1_intervals[h1_intervals[:, 1] < np.inf] if len(h1_intervals) > 0 else np.array([])
    return diag, h1_clean


diag_t, h1_t = get_tda_data(cloud_torus)


diag_s, h1_s = get_tda_data(cloud_sphere)

# --- CALCULATE GLOBAL LIMITS ---
# Combine all finite intervals to find the "window" of activity
all_finite = []
for d in [diag_t, diag_s]:
    for _, (b, d_val) in d:
        if d_val != np.inf:
            all_finite.append([b, d_val])
all_finite = np.array(all_finite)

f_min = np.min(all_finite)
f_max = np.max(all_finite)
margin = (f_max - f_min) * 0.05
limit_window = [f_min - margin, f_max + margin]

# ... [Keep your previous code for generating data and calculating limits] ...

# 3. Plotting 2x3 Grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- PLOTTING LOGIC ---
for row, (diag, h1, label) in enumerate([(diag_t, h1_t, "Torus"), (diag_s, h1_s, "Sphere")]):
    # Column 0: Barcode
    gudhi.plot_persistence_barcode(diag, axes=axes[row, 0], legend=True) # Added legend=True for consistency
    axes[row, 0].set_xlim(limit_window)
    axes[row, 0].set_title(f"{label}: Barcode", fontsize=18, weight="bold")
    # axes[row, 0].legend(loc="boc)

    # Column 1: Diagram
    gudhi.plot_persistence_diagram(diag, greyblock=False, axes=axes[row, 1], legend=True) # Added legend=True
    axes[row, 1].set_xlim(limit_window)
    axes[row, 1].set_ylim(limit_window)
    axes[row, 1].set_title(f"{label}: Diagram", fontsize=18, weight="bold")

    # Column 2: Landscape (H1)
    ls = Landscape(num_landscapes=5, resolution=100, sample_range=[f_min, f_max])

    l_vals = ls.fit_transform([h1]).reshape(5, 100) if len(h1) > 0 else np.zeros((5, 100))
    
    x_values = np.linspace(f_min, f_max, 100)
    for i in range(3):
        # Removed 'if row == 0' from label assignment so both rows have label data
        axes[row, 2].plot(x_values, l_vals[i], label=f"Layer {i+1}")
   
    axes[row, 2].set_xlim(limit_window)
    axes[row, 2].set_ylim(-0.05, 0.6) 
    axes[row, 2].set_title(f"{label}: H1 Landscape", fontsize=18, weight="bold")
    
    # Removed 'if row == 0' to ensure legend appears on BOTH rows
    axes[row, 2].legend(fontsize=14) 

plt.tight_layout()
savePic("All_together.png")
plt.show()


In [ ]:
fig = plt.figure(figsize=(7, 8), dpi=300) 
fig.set_size_inches(7.3, 8.2)
# Create a figure with two subplots

# Subplot 1: Noisy Sphere Points
ax2 = fig.add_subplot(211, projection='3d') # 1 row, 2 columns, first plot
ax2.scatter(cloud_sphere[:, 0], cloud_sphere[:, 1], cloud_sphere[:, 2], color='blue', alpha=0.7, s=10, label=f'Point Cloud {num_points} ')
# ax1.set_title(f'Noisy Point Cloud (Sphere Origin)')
ax2.set_xlabel('X', labelpad=-10)
ax2.set_ylabel('Y', labelpad=-10)
ax2.set_zlabel('Z', labelpad=-10)
ax2.set_xticklabels([])
ax2.set_yticklabels([])
ax2.set_zticklabels([])
# Set equal aspect ratio to prevent distortion
ax2.set_box_aspect([1, 1, 1])
ax2.legend(fontsize=12, prop={'weight': 'bold'})
ax2.grid(True)

# Subplot 2: Noisy Torus Points
ax1 = fig.add_subplot(212, projection='3d') # 1 row, 2 columns, second plot
ax1.scatter(cloud_torus[:, 0], cloud_torus[:, 1], cloud_torus[:, 2], color='red', alpha=0.7, s=10, label=f'Point Cloud {num_points}')
# ax2.set_title(f'Noisy Point Cloud (Torus Origin)')
ax1.set_xlabel('X', labelpad=-10)
ax1.set_ylabel('Y', labelpad=-10)
ax1.set_zlabel('Z', labelpad=-10)
ax1.set_xticklabels([])
ax1.set_yticklabels([])
ax1.set_zticklabels([])
# Set equal aspect ratio
ax1.set_box_aspect([1, 1, 1])
ax1.legend( fontsize=12, prop={'weight': 'bold'})
ax1.grid(True)
savePic("100 point clouds")
# plt.tight_layout() # Adjust subplot parameters for a tight layout

plt.show()

In [ ]:
# Resolution
res = 100
u = np.linspace(0, 2 * np.pi, res)
v = np.linspace(0, np.pi, res)
U, V = np.meshgrid(u, v)

# --- Sphere ---
x_s = 2 * np.sin(V) * np.cos(U)
y_s = 2 * np.sin(V) * np.sin(U)
z_s = 2 * np.cos(V)

# --- Torus ---
u_t = np.linspace(0, 2 * np.pi, res)
v_t = np.linspace(0, 2 * np.pi, res)
UT, VT = np.meshgrid(u_t, v_t)
R, r = 3, 1
x_t = (R + r * np.cos(VT)) * np.cos(UT)
y_t = (R + r * np.cos(VT)) * np.sin(UT)
z_t = 2 * r * np.sin(VT)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "surface"}, {"type": "surface"}]],
    subplot_titles=("Sphere", "Torus"),
)
fig.update_annotations(font_size=24)
green_gradient = [[0.0, "#d8f3dc"], [1.0, "#2d6a4f"]]
z_s_norm = (z_s - z_s.min()) / (z_s.max() - z_s.min())
z_t_norm = (z_t - z_t.min()) / (z_t.max() - z_t.min())

fig.add_trace(
    go.Surface(
        x=x_s, y=y_s, z=z_s,
        surfacecolor=z_s_norm,
        colorscale=green_gradient,
        showscale=False
    ), row=1, col=1
)

fig.add_trace(
    go.Surface(
        x=x_t, y=y_t, z=z_t,
        surfacecolor=z_t_norm,
        colorscale=green_gradient,
        showscale=False
    ), row=1, col=2
)

# Axis configuration for 3D: No tick labels. 
# Note: title_standoff is NOT supported in 3D scenes.
axis_style_s = dict(
    showticklabels=False, 
    range=[-2.5, 2.5],
    title=dict(text="Y")
)
axis_style_t = dict(
    showticklabels=False, 
    range=[-4.5, 4.5],
    title=dict(text="Y")
)

fig.update_layout(
    # 7.3 x 4.1 inches converted to pixels (700x394 approx)
    width=950,
    height=394,
    scene=dict(
        xaxis=axis_style_s,
        yaxis=dict(showticklabels=False, range=[-2.5, 2.5], title=dict(text="X")),
        zaxis=dict(showticklabels=False, range=[-2.5, 2.5], title=dict(text="Z")),
        aspectmode="cube"
    ),
    scene2=dict(
        xaxis=axis_style_t,
        yaxis=dict(showticklabels=False, range=[-4.5, 4.5], title=dict(text="X")),
        zaxis=dict(showticklabels=False, range=[-4.5, 4.5], title=dict(text="Z")),
        aspectmode="data"
    ),
    margin=dict(l=0, r=10, t=40, b=10),
)

fig.show()


In [ ]:
# Resolution: Increased to 400 for a perfectly smooth surface
res = 400
u = np.linspace(0, 2 * np.pi, res)
v = np.linspace(0, np.pi, res)
U, V = np.meshgrid(u, v)

# --- Sphere ---
x_s = 2 * np.sin(V) * np.cos(U)
y_s = 2 * np.sin(V) * np.sin(U)
z_s = 2 * np.cos(V)

# --- Torus ---
u_t = np.linspace(0, 2 * np.pi, res)
v_t = np.linspace(0, 2 * np.pi, res)
UT, VT = np.meshgrid(u_t, v_t)
R, r = 3, 1
x_t = (R + r * np.cos(VT)) * np.cos(UT)
y_t = (R + r * np.cos(VT)) * np.sin(UT)
z_t = 1.2 * r * np.sin(VT)

# Setup figure (7.3 x 4.1 inches)
fig = plt.figure(figsize=(7.3, 4.1), dpi=150)
cmap = plt.cm.Greens

# --- Plot 1: Sphere ---
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
# rcount and ccount set to 'res' force high-detail sampling
surf1 = ax1.plot_surface(x_s, y_s, z_s, cmap=cmap, linewidth=0, 
                         antialiased=False, shade=True, rcount=res, ccount=res)
ax1.set_title("Sphere", fontsize=14, pad=20, weight='bold' )
ax1.set_xticklabels([]); ax1.set_yticklabels([]); ax1.set_zticklabels([])
ax1.set_xlabel('X', labelpad=-10); ax1.set_ylabel('Y', labelpad=-10); ax1.set_zlabel('Z', labelpad=-10)
ax1.set_box_aspect([1,1,1])

# --- Plot 2: Torus ---
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
surf2 = ax2.plot_surface(x_t, y_t, z_t, cmap=cmap, linewidth=0, 
                         antialiased=False, shade=True, rcount=res, ccount=res)
ax2.set_title("Torus", fontsize=16, pad=20, weight="bold")
ax2.set_xticklabels([]); ax2.set_yticklabels([]); ax2.set_zticklabels([])
ax2.set_xlabel('X', labelpad=-10); ax2.set_ylabel('Y', labelpad=-10); ax2.set_zlabel('Z', labelpad=-10)
ax2.set_box_aspect([np.ptp(x_t), np.ptp(y_t), np.ptp(z_t)])

# plt.tight_layout()
plt.show()


In [ ]:
# Resolution: 500 for high surface smoothness
res = 500 

# --- Torus (Major R=3, Minor r=1) ---
u_t = np.linspace(0, 2 * np.pi, res)
v_t = np.linspace(0, 2 * np.pi, res)
UT, VT = np.meshgrid(u_t, v_t)
R, r = 3, 1

x_t = (R + r * np.cos(VT)) * np.cos(UT)
y_t = (R + r * np.cos(VT)) * np.sin(UT)
z_t = r * np.sin(VT)

fig = go.Figure(data=[go.Surface(
    z=z_t, x=x_t, y=y_t, 
    colorscale='Blues', 
    opacity=0.9, 
    showscale=False,
    contours={
        "x": {"show": True, "color": "white", "width": 1},
        "y": {"show": True, "color": "white", "width": 1},
        "z": {"show": True, "color": "white", "width": 1},
    }
)])

# Axis settings to hide ticks and markers
no_axis = dict(
    showgrid=False, 
    zeroline=False, 
    showline=False, 
    showticklabels=False, 
    title='',
    visible=False # This removes the surrounding box entirely
)

fig.update_layout(
    title={
        'text': "Torus with Main Loops",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=48, weight="bold") # Very large title
    },
    width=1200, 
    height=800,
    scene=dict(
        aspectmode='data',
        xaxis=no_axis,
        yaxis=no_axis,
        zaxis=no_axis,
    ),
    margin=dict(l=0, r=0, b=0, t=100) # Reduces empty space
)

fig.show()



In [ ]:
# Resolution
res = 200 # High res for the base surface
u = np.linspace(0, 2 * np.pi, res)
v = np.linspace(0, 2 * np.pi, res)
U, V = np.meshgrid(u, v)

# --- Torus Parameters ---
R, r = 3, 1
x = (R + r * np.cos(V)) * np.cos(U)
y = (R + r * np.cos(V)) * np.sin(U)
z = r * np.sin(V)

# Setup Figure (7.3 x 4.1 inches)
fig = plt.figure(figsize=(7.3, 4.1), dpi=150)
ax = fig.add_subplot(111, projection='3d')

# Plot the smooth surface
# rstride/cstride control how often the "white lines" (grid) appear
surf = ax.plot_surface(x, y, z, cmap='Blues', 
                       linewidth=0.5,      # Width of the contour lines
                       edgecolor='white',  # Color of the "main loops"
                       antialiased=True, 
                       shade=True,
                       rcount=20,          # Number of loops in one direction
                       ccount=20)          # Number of loops in the other direction

# Bold Title using set_title
ax.set_title("Torus with Main Loops", fontsize=14, fontweight='bold', pad=20)

# Hide axes, panes, and ticks for a clean look
ax.set_axis_off()

# Set equal aspect ratio
ax.set_box_aspect([np.ptp(x), np.ptp(y), np.ptp(z)])

plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.patches import Rectangle
# ------------------------------------------------------------
# Canvas
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 12)
ax.set_ylim(2, 8)
ax.axis("off")
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
LINE = "#4d4d4d"
TEXT = "#5a5a5a"
ARROW = "#7a7a7a"
BLUE = "#79c9d3"
RED = "#d97a7a"
TEAL = "#8bc4c9"
GREEN = "#9bcf7a"

# ------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------
def pentagon_points(center, r=0.53):
    """
    Slightly irregular upright pentagon matching the sketch style.
    Vertex order goes around the boundary.
    """
    cx, cy = center
    angles = np.deg2rad([118, 182, 252, 338, 32])
    pts = np.column_stack([cx + r * np.cos(angles), cy + r * np.sin(angles)])
    return pts

def draw_vertices(ax, pts, color=LINE, size=18):
    ax.scatter(pts[:, 0], pts[:, 1], s=size, color=color, zorder=5)

def draw_edges(ax, pts, edges, color=LINE, lw=2.2):
    for i, j in edges:
        ax.plot([pts[i, 0], pts[j, 0]], [pts[i, 1], pts[j, 1]],
                color=color, lw=lw, solid_capstyle="round", zorder=3)

def draw_triangle_fill(ax, pts, tri, color, alpha=0.35, hatch="////"):
    poly = Polygon(
        pts[list(tri)],
        closed=True,
        facecolor=color,
        edgecolor=color,
        linewidth=0.8,
        hatch=hatch,
        alpha=alpha,
        zorder=1
    )
    ax.add_patch(poly)

def boundary_edges(n=5):
    return [(i, (i + 1) % n) for i in range(n)]

def triangle_edges(tri):
    a, b, c = tri
    return [(a, b), (b, c), (a, c)]

def unique_undirected_edges(edges):
    out = []
    seen = set()
    for i, j in edges:
        e = tuple(sorted((i, j)))
        if e not in seen:
            seen.add(e)
            out.append(e)
    return out

def draw_open_chain(ax, center, r=0.53):
    pts = pentagon_points(center, r)
    open_edges = [(0,1), (1,2), (2,3), (3,4)]
    draw_edges(ax, pts, open_edges)
    draw_vertices(ax, pts)
    return pts

def draw_cycle(ax, center, r=0.53):
    pts = pentagon_points(center, r)
    draw_edges(ax, pts, boundary_edges())
    draw_vertices(ax, pts)
    return pts

def draw_cycle_with_fills(ax, center, fills, r=0.53):
    """
    fills: list of (triangle_indices, color)
    Triangles are filled first, then all needed edges are drawn.
    """
    pts = pentagon_points(center, r)

    for tri, color in fills:
        draw_triangle_fill(ax, pts, tri, color)

    edges = boundary_edges()
    for tri, _ in fills:
        edges.extend(triangle_edges(tri))
    edges = unique_undirected_edges(edges)

    draw_edges(ax, pts, edges)
    draw_vertices(ax, pts)
    return pts

def light_blue_arrow(ax, x, y, dx=0.28, dy=0):
    arr = FancyArrowPatch(
        (x, y), (x + dx, y + dy),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.5,
        color=BLUE,
        zorder=6
    )
    ax.add_patch(arr)

def grey_arrow(ax, x1, y1, x2, y2):
    arr = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=16,
        linewidth=1.8,
        color=ARROW,
        zorder=2
    )
    ax.add_patch(arr)


# Style Colors (Keep your existing palette)
LINE, TEXT, ARROW, BLUE, RED, TEAL, GREEN = "#4d4d4d", "#5a5a5a", "#7a7a7a", "#79c9d3", "#d97a7a", "#8bc4c9", "#9bcf7a"

# ------------------------------------------------------------
# NEW: Radius Bar (Replacing algebraic sequence)
# ------------------------------------------------------------
bar_y = 7.0
bar_start_x = 3.0   # Matches "Loop forms" column
bar_end_x = 11.25   # Matches "loop dies" column

# Draw the main timeline/axis for radius
ax.add_patch(FancyArrowPatch((1.0, bar_y), (12.5, bar_y), arrowstyle="-|>", 
                             mutation_scale=20, linewidth=2, color=LINE, zorder=1))
ax.plot([1.0, 13.0], [bar_y, bar_y], color=LINE, lw=1.5, zorder=1)
ax.text(0.8, bar_y, "r", fontsize=30, color=TEXT, fontweight='bold', ha='right', va='center')

# Draw the "Persistence Bar"
# This represents the lifetime of the topological feature
persistence_bar = Rectangle(
    (bar_start_x, bar_y - 0.15), 
    bar_end_x - bar_start_x, 0.3, 
    color=BLUE, alpha=0.8, zorder=2, label="Feature Persistence"
)
ax.add_patch(persistence_bar)

# Add tick marks for clarity
ax.plot([bar_start_x, bar_start_x], [bar_y - 0.3, bar_y + 0.3], color=LINE, lw=2)
ax.plot([bar_end_x, bar_end_x], [bar_y - 0.3, bar_y + 0.3], color=LINE, lw=2)

ax.text(6.5, bar_y + 0.6, "Increasing Radius", fontsize=22, color=TEXT, ha="center", weight="bold" )

# Draw the "Persistence Bar"
# This represents the lifetime of the topological feature

# ------------------------------------------------------------
# Middle diagram row
# ------------------------------------------------------------
shape_y = 4.9
shape_xs = [1.0, 3.0, 5.0, 7.0, 9.1, 11.25]

# Vertex numbering around the pentagon:
# 0 = upper-left
# 1 = left
# 2 = lower-left
# 3 = lower-right
# 4 = upper-right

# 1) no cycle
draw_open_chain(ax, (shape_xs[0], shape_y))

# 2) cycle forms
draw_cycle(ax, (shape_xs[1], shape_y))

# 3) cycle persists (one triangle)
draw_cycle_with_fills(
    ax, (shape_xs[2], shape_y),
    fills=[((0, 1, 4), RED)]
)

# 4) cycle persists (same class)
draw_cycle_with_fills(
    ax, (shape_xs[3], shape_y),
    fills=[((0, 1, 4), RED)]
)

# 5) cycle persists (two triangles sharing only one edge, no overlap)
draw_cycle_with_fills(
    ax, (shape_xs[4], shape_y),
    fills=[
        ((0, 1, 4), TEAL),
        ((1, 2, 3), RED),
    ]
)

# 6) cycle dies (three adjacent triangles, edge-sharing only, no overlap)
draw_cycle_with_fills(
    ax, (shape_xs[5], shape_y),
    fills=[
        ((0, 1, 4), TEAL),   # top-left / top-right
        ((1, 2, 3), GREEN),  # lower-left band
        ((1, 3, 4), RED),    # right-side triangle
    ]
)

# little blue arrows above the evolving shapes
for x in shape_xs[1:]:
    light_blue_arrow(ax, x - 1.2 , shape_y)

# ------------------------------------------------------------
# Labels under shapes
# ------------------------------------------------------------
ax.text(shape_xs[0], 4.12, "no loop", ha="center", va="center",
        fontsize=22, color=TEXT)

ax.text(shape_xs[1], 3.98, "Loop\nforms", ha="center", va="center",
        fontsize=22, color=TEXT, linespacing=0.9)

ax.text(shape_xs[5], 4.04, "loop\ndies", ha="center", va="center",
        fontsize=22, color=TEXT, linespacing=0.9)

# "cycle persists" underline arrow
ax.plot([4.78, 8.55], [3.58, 3.58], color=ARROW, lw=1.6, zorder=2)
grey_arrow(ax, 5.1, 3.53, 8.52, 3.53)
ax.text(6.68, 3.1, "loop persists", ha="center", va="center",
        fontsize=30, color=TEXT)

plt.tight_layout()
savePic("replicated_homology_sequence_no_overlap.png")
plt.show()

# To save:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Setup the figure with a weighted grid
fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 6, wspace=0.4)

# Global Style Parameters
LW = 4            # Line Width
PS = 100        # Point Size
FC = '#9ACD32'    # Face Color (Lime Green)
EC = '#3333CC'    # Edge Color (Blue)
LABEL_Y = 0.08    # Global Y position for labels at the very bottom

# --- 0-Simplex (Vertex) ---
ax0 = fig.add_subplot(gs[0, 0])
ax0.scatter([0.5], [0.5], color='black', s=PS, zorder=5)
ax0.set_axis_off()
fig.text(0.18, LABEL_Y, "0-simplex", ha='center', fontsize=18, fontweight='bold')

# --- 1-Simplex (Edge) ---
ax1 = fig.add_subplot(gs[0, 1])
ax1.plot([0.1, 0.9], [0.5, 0.5], color=EC, lw=LW)
ax1.scatter([0.1, 0.9], [0.5, 0.5], color='black', s=PS, zorder=5)
ax1.set_axis_off()
fig.text(0.30, LABEL_Y, "1-simplex", ha='center', fontsize=18, fontweight='bold')

# --- 2-Simplex (Equilateral Triangle) ---
ax2 = fig.add_subplot(gs[0, 2])
# Geometric Calculation for Equilateral
side = 0.8
h = (np.sqrt(3)/2) * side  # Height ≈ 0.69
y_start = 0.15
tri_pts = np.array([
    [0.5 - side/2, y_start], 
    [0.5 + side/2, y_start], 
    [0.5, y_start + h -.2]
]) 
tri = plt.Polygon(tri_pts, fc=FC, ec=EC, lw=LW, alpha=0.9)
ax2.add_patch(tri)
ax2.scatter(tri_pts[:,0], tri_pts[:,1], color='black', s=PS, zorder=5)
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.set_axis_off()
fig.text(0.44, LABEL_Y, "2-simplex", ha='center', fontsize=18, fontweight='bold')

# --- 3-Simplex (Short Tetrahedron) ---
ax3 = fig.add_subplot(gs[0, 3], projection='3d')
v = np.array([[0.1, 0.1, 0.1], [0.9, 0.1, 0.1], [0.5, 0.8, 0.1], [0.5, 0.4, 0.5]])
faces = [[v[0], v[1], v[3]], [v[1], v[2], v[3]], [v[2], v[0], v[3]], [v[0], v[1], v[2]]]
ax3.add_collection3d(Poly3DCollection(faces, facecolors=FC, linewidths=LW, edgecolors=EC, alpha=0.7))
ax3.scatter(v[:,0], v[:,1], v[:,2], color='black', s=PS/2, depthshade=False)
ax3.set_axis_off()
ax3.set_zlim(0, 0.8)
fig.text(0.58, LABEL_Y, "3-simplex", ha='center', fontsize=18, fontweight='bold')

# --- Simplicial Complex (Wide Plot with Border) ---
ax4 = fig.add_subplot(gs[0, 4:])
c_pts = np.array([[0.05, 0.6], [0.15, 0.3], [0.4, 0.35], [0.25, 0.7], [0.55, 0.2], [0.7, 0.4], [0.9, 0.25], [0.95, 0.7]])
t1 = plt.Polygon(c_pts[0:3], fc=FC, ec=EC, lw=LW, alpha=0.8)
t2 = plt.Polygon(c_pts[4:7], fc=FC, ec=EC, lw=LW, alpha=0.8)
ax4.add_patch(t1); ax4.add_patch(t2)
ax4.plot(c_pts[2:5, 0], c_pts[2:5, 1], color=EC, lw=LW)
ax4.plot(c_pts[6:8, 0], c_pts[6:8, 1], color=EC, lw=LW)
ax4.scatter(c_pts[:,0], c_pts[:,1], color='black', s=PS, zorder=5)

# Styling the Complex Box
ax4.set_xticks([]); ax4.set_yticks([])
ax4.set_xlim(0, 1); ax4.set_ylim(0, 1)
for spine in ax4.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
fig.text(0.78, LABEL_Y, "Simplicial Complex", ha='center', fontsize=18, fontweight='bold')

plt.subplots_adjust(bottom=0.25)
savePic("simplex.png")
plt.show()


In [ ]:
import ipywidgets as widgets
from scipy.spatial import cKDTree
# --- 1. Generate Vertical Elongated 8-Shape (15 Points) ---
n_points = 15
t = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
points_2d = []
noise_scale = 0.1

for val in t:
    y = 5 * np.sin(val)
    x = 2 * np.sin(val) * np.cos(val)
    x_n = x + np.random.normal(0, noise_scale)
    y_n = y + np.random.normal(0, noise_scale)
    points_2d.append([y_n, x_n])

points_2d = np.array(points_2d)

# --- 2. Geometric Intersection Helper ---
def intersect(a, b, c, d):
    def ccw(A, B, C):
        return (C[1]-A[1]) * (B[0]-A[0]) > (B[1]-A[1]) * (C[0]-A[0])
    if any(np.array_equal(a, x) or np.array_equal(b, x) for x in [c, d]):
        return False
    return ccw(a,c,d) != ccw(b,c,d) and ccw(a,b,c) != ccw(a,b,d)

# --- 3. Visualization Function ---
def update_plot(radius):
    plt.close('all')
    fig, ax = plt.subplots(figsize=(6, 10))
    
    for p in points_2d:
        circ = plt.Circle((p[0], p[1]), radius, color='blue', alpha=0.07, lw=0)
        ax.add_patch(circ)

    tree = cKDTree(points_2d)
    pairs = list(tree.query_pairs(radius))
    dist_pairs = sorted([(np.linalg.norm(points_2d[i] - points_2d[j]), i, j) for i, j in pairs])
    
    valid_edges = []
    # Set for fast lookup of existing connections
    edge_set = set()

    for _, i, j in dist_pairs:
        p1, p2 = points_2d[i], points_2d[j]
        if not any(intersect(p1, p2, points_2d[e1], points_2d[e2]) for e1, e2 in valid_edges):
            valid_edges.append((i, j))
            edge_set.add(tuple(sorted((i, j))))
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='red', lw=1.5, alpha=0.8, zorder=2)

    # --- Triangle Shading Logic ---
    # Loop through all triplets to see if they form a closed triangle
    for i in range(n_points):
        for j in range(i + 1, n_points):
            for k in range(j + 1, n_points):
                # Check if all 3 edges of the triangle (i,j,k) exist in valid_edges
                if (tuple(sorted((i, j))) in edge_set and 
                    tuple(sorted((j, k))) in edge_set and 
                    tuple(sorted((i, k))) in edge_set):
                    
                    # Extract coordinates for the 3 points
                    tri_coords = points_2d[[i, j, k]]
                    ax.fill(tri_coords[:, 0], tri_coords[:, 1], color='cyan', alpha=0.4, zorder=1)

    ax.scatter(points_2d[:,0], points_2d[:,1], c='darkblue', s=40, zorder=10)
    ax.set_aspect('equal')
    ax.set_xlim(-7, 7)
    ax.set_ylim(-4, 4)
    ax.set_title(f"Figure-Eight: Radius {radius:.2f}", fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.6)
    savePic("twoloops.png")
    plt.show()

# --- 4. Interface ---
radius_slider = widgets.FloatSlider(
    value=1.5, min=0.1, max=6.0, step=0.1, 
    description='Growth Radius:',
    style={'description_width': 'initial'}
)
output = widgets.interactive_output(update_plot, {'radius': radius_slider})
display(radius_slider, output)

points_2 = points_2d

print(f"Loaded {len(points_2)} points from /content/drive/MyDrive/TDA Project/data sample 2.csv")
plt.figure(figsize=(8, 6))
plt.scatter(points_2[:, 0], points_2[:, 1], alpha=0.7, color='blue')
plt.title('Scatter Plot of Loaded Points')
plt.xlabel('X-coordinate')
plt.ylabel('Y-coordinate')
plt.grid(True)
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# 2. Build a Rips complex
rips = gudhi.RipsComplex(points=points_2, max_edge_length=6)

# 3. Create a simplex tree
st = rips.create_simplex_tree(max_dimension=2)

# 4. Compute persistence and store the result
persistence_2 = st.persistence()

# # 5. Extract and print the persistence intervals
# print("Persistence intervals:")
# for dim, (birth, death) in persistence:
#     print(f"Dimension {dim}: [{birth:.4f}, {death:.4f})")

# --- 7. Plot Barcode ---
# Enable legend=True to show homology dimension colors
gudhi.plot_persistence_barcode(persistence_2, legend=True)
plt.title("Persistence Barcode", fontsize=18, weight="bold")
savePic("Barcode.png")
plt.show()

# --- 6. Plot the Persistence Diagram ---
# Use 'greyblock=False' to remove the lower gray area
# Set legend=True to match the barcode's legend
gudhi.plot_persistence_diagram(persistence_2, greyblock=False, legend=True)
plt.title("Persistence Diagram", fontsize=18, weight="bold")
plt.grid(True, linestyle='--', alpha=0.6)
savePic("Digs.png")
plt.show()




In [ ]:

# Disable LaTeX to avoid the dvipng warning
plt.rcParams['text.usetex'] = False


def get_tda_data(cloud):
    rips = gudhi.RipsComplex(points=cloud)
    st = rips.create_simplex_tree(max_dimension=2)
    diag = st.persistence()
    h1_intervals = st.persistence_intervals_in_dimension(1)
    # Filter infinite values for landscape/axis calculations
    h1_clean = h1_intervals[h1_intervals[:, 1] < np.inf] if len(h1_intervals) > 0 else np.array([])
    return diag, h1_clean


diag_t, h1_t = get_tda_data(cloud_torus)


diag_s, h1_s = get_tda_data(cloud_sphere)

# --- CALCULATE GLOBAL LIMITS ---
# Combine all finite intervals to find the "window" of activity
all_finite = []
for d in [diag_t, diag_s]:
    for _, (b, d_val) in d:
        if d_val != np.inf:
            all_finite.append([b, d_val])
all_finite = np.array(all_finite)

f_min = np.min(all_finite)
f_max = np.max(all_finite)
margin = (f_max - f_min) * 0.05
limit_window = [f_min - margin, f_max + margin]

# ... [Keep your previous code for generating data and calculating limits] ...

# 3. Plotting 2x3 Grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- PLOTTING LOGIC ---
for row, (diag, h1, label) in enumerate([(diag_t, h1_t, "Torus"), (diag_s, h1_s, "Sphere")]):
    # Column 0: Barcode
    gudhi.plot_persistence_barcode(diag, axes=axes[row, 0], legend=True) # Added legend=True for consistency
    axes[row, 0].set_xlim(limit_window)
    axes[row, 0].set_title(f"{label}: Barcode", fontsize=18, weight="bold")
    # axes[row, 0].legend(loc="boc)

    # Column 1: Diagram
    gudhi.plot_persistence_diagram(diag, greyblock=False, axes=axes[row, 1], legend=True) # Added legend=True
    axes[row, 1].set_xlim(limit_window)
    axes[row, 1].set_ylim(limit_window)
    axes[row, 1].set_title(f"{label}: Diagram", fontsize=18, weight="bold")

    # Column 2: Landscape (H1)
    ls = Landscape(num_landscapes=5, resolution=100, sample_range=[f_min, f_max])
    l_vals = ls.fit_transform([h1]).reshape(5, 100) if len(h1) > 0 else np.zeros((5, 100))
    
    x_values = np.linspace(f_min, f_max, 100)
    for i in range(3):
        # Removed 'if row == 0' from label assignment so both rows have label data
        axes[row, 2].plot(x_values, l_vals[i], label=f"Layer {i+1}")
   
    axes[row, 2].set_xlim(limit_window)
    axes[row, 2].set_ylim(-0.05, 0.6) 
    axes[row, 2].set_title(f"{label}: H1 Landscape", fontsize=18, weight="bold")
    
    # Removed 'if row == 0' to ensure legend appears on BOTH rows
    axes[row, 2].legend(fontsize=14) 

plt.tight_layout()
savePic("All_together.png")
plt.show()


In [ ]:
treatment = pd.read_csv("./workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("./workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")

x_treatment = treatment['Center of the object_0']
y_treatment = treatment['Center of the object_1']
z_treatment = treatment['Center of the object_2']

x_control = control['Center of the object_0']
y_control = control['Center of the object_1']
z_control = control['Center of the object_2']

treatment_data = np.column_stack((x_treatment, y_treatment, z_treatment))
control_data = np.column_stack((x_control, y_control, z_control))

dgm_treatment_data   = Rips(maxdim=2).fit_transform(treatment_data)
dgm_control_data     = Rips(maxdim=2).fit_transform(control_data)


fig = plt.figure(figsize=(12,8), dpi=300)
plt.rcParams["font.weight"] = "bold"
# Plot Data

#3D plots
ax0 = fig.add_subplot(2, 3, 1, projection='3d')
ax1 = fig.add_subplot(2, 3, 4, projection='3d')

ax1.scatter(treatment_data[:, 0], treatment_data[:, 1], treatment_data[:,2])
ax1.set_title("Treatment")
ax1.view_init(elev=25, azim=45, roll=0)
# Change rotation: 30 degrees elevation, 45 degrees azimuth
ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:,2])
ax0.view_init(elev=25, azim=45, roll=0)
ax0.set_title("Control")
# Plot Persistence Diagrams
# fig2 = plt.figure(figsize=(10,10), dpi=300)
ax2 = fig.add_subplot(2, 3, 2)
ax3 = fig.add_subplot(2, 3, 5)


plot_diagrams(dgm_control_data, title="Control", ax=ax2)
plot_diagrams(dgm_treatment_data, title="Treatment", ax=ax3)


#3D plots
ax4 = fig.add_subplot(2, 3, 3)
ax5 = fig.add_subplot(2, 3, 6)


pl_treatment = PersLandscapeExact(dgm_treatment_data, hom_deg=1)
pl_control = PersLandscapeExact(dgm_control_data, hom_deg=1)

plot_landscape_simple(pl_treatment, ax=ax4, depth_range=range(4), title=r"Peristence Landscape $H_1$")
plot_landscape_simple(pl_control, ax=ax5, depth_range=range(4), title=r"Persistence Landscape $H_1$")
ax4.set_xlim(30,110)
ax4.set_ylim(-2,25)
ax5.set_xlim(30,110)
ax5.set_ylim(-2,25)
fig.tight_layout()


<h1> Using Gudhi instead of Persim</h1>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import gudhi
import gudhi.representations

# --- Data Loading and Standardization ---
treatment = pd.read_csv("./Point clouds-selected/centroids Epha2a hom_table.csv")
control = pd.read_csv("./Point clouds-selected/centroids control_table.csv")

# Extract coordinates
coords = ['Center of the object_0', 'Center of the object_1', 'Center of the object_2']
treatment_raw = treatment[coords].values
control_raw = control[coords].values

# Standardize the data before applying GUDHI
scaler = StandardScaler()
treatment_data = scaler.fit_transform(treatment_raw)
control_data = scaler.fit_transform(control_raw)

def get_gudhi_persistence(data):
    # No max_edge_length provided; GUDHI defaults to infinity
    rips = gudhi.RipsComplex(points=data)
    simplex_tree = rips.create_simplex_tree(max_dimension=2)
    return simplex_tree

treatment_data_simplex_tree = get_gudhi_persistence(treatment_data)
control_data_simplex_tree = get_gudhi_persistence(control_data)

fig = plt.figure(figsize=(18, 10), dpi=150)

def set_bold_title(ax, title):
    ax.set_title(title, fontweight='bold', fontsize=18)

# --- 1. 3D Plots (Standardized Data) ---
ax0 = fig.add_subplot(2, 3, 1, projection='3d')
ax1 = fig.add_subplot(2, 3, 4, projection='3d')

ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2])
ax0.view_init(elev=25, azim=45)
set_bold_title(ax0, "Control")

ax1.scatter(treatment_data[:, 0], treatment_data[:, 1], treatment_data[:, 2])
ax1.view_init(elev=25, azim=45)
set_bold_title(ax1, "Treatment")

# --- 2. Persistence Diagrams (No Gray Block) ---
ax2 = fig.add_subplot(2, 3, 2)
ax3 = fig.add_subplot(2, 3, 5)

for ax, dgm, label in zip([ax2, ax3], [control_data_simplex_tree.persistence(), treatment_data_simplex_tree.persistence()], ["Control", "Treatment"]):
    gudhi.plot_persistence_diagram(dgm, axes=ax, legend=True)
    # Remove the gray diagonal shading (gray block)
    leg = ax.get_legend()
    if leg:
        for text in leg.get_texts():
            text.set_fontsize(18)
    for poly in ax.findobj(plt.Polygon):
        poly.set_visible(False)
    set_bold_title(ax, f"{label}: Persistence Diagram ")

# --- 3. Persistence Landscapes (Default Resolution, Independent Axes) ---
ax4 = fig.add_subplot(2, 3, 3)
ax5 = fig.add_subplot(2, 3, 6)

num_layers = 4
resolution = 1000 # Default GUDHI resolution
LS = gudhi.representations.Landscape(num_landscapes=num_layers, resolution=resolution)

# Filter for H1 points (dimension 1)
h1_control = control_data_simplex_tree.persistence_intervals_in_dimension(1)
h1_treatment = treatment_data_simplex_tree.persistence_intervals_in_dimension(1)

for ax, h1_data, label in zip([ax4, ax5], [h1_control, h1_treatment], ["Control", "Treatment"]):
    if len(h1_data) > 0:
        lsc = LS.fit_transform([h1_data]).reshape(num_layers, -1)
        # Independent x-axis based on birth/death range
        x_min, x_max = h1_data[:, 0].min(), h1_data[:, 1].max()
        x_vals = np.linspace(x_min, x_max, resolution)
        for i in range(num_layers):
            ax.plot(x_vals, lsc[i], label=f'Layer {i+1}')
        ax.legend(fontsize=16)
    set_bold_title(ax, f"{label}: H1 Landscape")

fig.tight_layout()
for axes in [ax2, ax3]:
    axes.set_xlim(-0.2, 1.2)
    axes.set_ylim(-0.2, 1.5)
for axes in [ax4, ax5]:
     axes.set_xlim(0.1, 1.3)
     # axes.set_ylim(-0.2, 1.5)
ax2.axhline(np.max(h1_control)*1.1, color="black")
ax3.axhline(np.max(h1_treatment)*1.1, color="black")
x_values = np.linspace(-0.2, 1.5)
ax2.plot(x_values, x_values, color="black")
ax3.plot(x_values, x_values, color="black")
# savePic("ApplicationInfoRow.png")
plt.show()


In [ ]:
import qrcode

def generate_qr(link, filename="qrcodo.png"):
    """
    Generates a QR code from a link and saves it as an image.
    """
    # Initialize the QR code object
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    
    # Add the link data
    qr.add_data(link)
    qr.make(fit=True)

    # Create the image (black QR on white background)
    img = qr.make_image(fill_color="black", back_color="white")
    
    # Save the file
    img.save(filename)
    return f"QR code successfully saved as {filename}"

# Example Usage
print(generate_qr( "https://jumpshare.com/share/dAnuvXbawl2OmYmOpkYg", "qr_figure_eight.png" ))
print(generate_qr("https://jumpshare.com/share/ZmkPC0q2ZQm12e3S3cAx", "qr_Torus.png" ))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import Delaunay
from scipy.spatial.distance import pdist, squareform
import imageio.v2 as imageio
import os

# --- Configuration ---
n_points = 100
total_frames = 200
max_eps = 5.0
R, r = 5, 2
output_folder = "tda_pure_torus"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- 1. Generate Points ---
theta = np.random.uniform(0, 2 * np.pi, n_points)
phi = np.random.uniform(0, 2 * np.pi, n_points)
points_3d = np.stack([
    (R + r * np.cos(phi)) * np.cos(theta),
    (R + r * np.cos(phi)) * np.sin(theta),
    r * np.sin(phi)
], axis=1)

tri = Delaunay(points_3d)

# H1 barcode: TWO primary loops + noise
h1_bars = [(0.8, 3.3), (1.2, 3.0)] + \
          [(np.random.uniform(0.4, 1.1), np.random.uniform(1.2, 1.5)) for _ in range(35)]

# UPDATED LABELS
labels = {0: "Major loop", 1: "Minor loop"}

# --- 2. Rendering ---
frame_paths = []
for frame_idx in range(total_frames):
    fig = plt.figure(figsize=(18, 6), dpi=150)
    ax_3d = fig.add_subplot(131, projection='3d')
    ax_bar = fig.add_subplot(132)
    ax_diag = fig.add_subplot(133)

    current_eps = (frame_idx / total_frames) * max_eps
    rot_angle = (frame_idx / total_frames) * 360

    # A. 3D Torus Plot
    faces = []
    for s in tri.simplices:
        for f in [(0,1,2), (0,1,3), (0,2,3), (1,2,3)]:
            face_pts = points_3d[s[list(f)]]
            dists = [np.linalg.norm(face_pts[i]-face_pts[j]) for i,j in [(0,1),(1,2),(2,0)]]
            if max(dists) <= current_eps:
                faces.append(face_pts)

    if faces:
        poly = Poly3DCollection(faces, facecolors='skyblue', edgecolors='red', alpha=0.2, linewidths=0.1)
        ax_3d.add_collection3d(poly)

    ax_3d.scatter(points_3d[:,0], points_3d[:,1], points_3d[:,2], c='darkblue', s=5, alpha=0.7)
    ax_3d.set_xlim(-8, 8); ax_3d.set_ylim(-8, 8); ax_3d.set_zlim(-4, 4)
    ax_3d.view_init(elev=45, azim=rot_angle)
    ax_3d.set_axis_off() 
    ax_3d.set_title(f"Torus ε = {current_eps:.2f}")

    # B. Persistence Barcode
    for i, (birth, death) in enumerate(h1_bars):
        if current_eps > birth:
            end_val = min(current_eps, death)
            color = 'crimson' if i < 2 else 'teal'
            # Added zorder to lines
            ax_bar.hlines(i, birth, end_val, colors=color, lw=2.5 if i < 2 else 0.5, alpha=0.8, zorder=1)
            
            # UPDATED TEXT: BLACK COLOR AND ON TOP (zorder)
            if i in labels:
                ax_bar.text(birth - .5 , i + 0.5, labels[i], color='black', fontsize=9, fontweight='bold', zorder=10)
    
    ax_bar.set_xlim(0, 4) 
    ax_bar.set_ylim(-1, len(h1_bars))
    ax_bar.set_title("Barcode ($H_1$ Features)")
    ax_bar.set_yticks([])

    # C. Persistence Diagram
    ax_diag.plot([0, max_eps], [0, max_eps], 'k--', alpha=0.3)
    for i, (birth, death) in enumerate(h1_bars):
        if current_eps >= birth:
            color = 'crimson' if i < 2 else 'teal'
            ax_diag.scatter(birth, min(current_eps, death), color=color, s=25, alpha=0.7)
    
    ax_diag.set_xlim(0, 4)
    ax_diag.set_ylim(0, max_eps)
    ax_diag.set_title("Birth vs Death")
    ax_diag.grid(True, linestyle=':', alpha=0.5)

    plt.tight_layout()
    path = f"{output_folder}/f_{frame_idx:03d}.png"
    plt.savefig(path)
    frame_paths.append(path)
    plt.close()

# --- 3. Save as GIF & Cleanup ---
images = [imageio.imread(f) for f in frame_paths]
imageio.mimsave('torus_pure_tda.gif', images, fps=12, loop=0)

for f in frame_paths:
    os.remove(f)
os.rmdir(output_folder)

print("Done! Saved as torus_pure_tda.gif")
